# Ch 7. LLM으로 KG 구축 — 엔티티·관계 추출

지식 그래프와 GraphRAG 스터디 · [study-graphrag](https://desty.github.io/study-graphrag/)

본문 [Ch 7](https://desty.github.io/study-graphrag/part3/07-llm-extraction/)의 코드입니다. `ANTHROPIC_API_KEY`가 필요하고, 적재 셀은 Ch 6의 Neo4j 연결을 가정합니다.

In [ ]:
!pip -q install anthropic
import os, json, anthropic
# os.environ['ANTHROPIC_API_KEY'] = '...'  # Colab Secrets 사용 권장
client = anthropic.Anthropic()

ONTOLOGY = {
    "entities": ["Drug", "Disease", "Ingredient"],
    "relations": ["treats", "contains", "interactsWith"],
}

In [ ]:
PROMPT = """다음 텍스트에서 트리플을 추출해 JSON으로만 답하라.
허용 엔티티 타입: {entities}
허용 관계 타입: {relations}
규칙: 허용 목록에 없는 타입은 만들지 말 것. 불확실하면 빼라.
형식: {{"triples": [{{"subj": "...", "subj_type": "...", "rel": "...", "obj": "...", "obj_type": "..."}}]}}

텍스트:
{text}"""

def extract(text):
    msg = client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=1024,
        messages=[{"role": "user", "content": PROMPT.format(
            entities=ONTOLOGY["entities"], relations=ONTOLOGY["relations"], text=text)}],
    )
    return json.loads(msg.content[0].text)

out = extract("아스피린은 두통을 완화하며, 성분으로 아세틸살리실산을 포함한다.")
print(json.dumps(out, ensure_ascii=False, indent=2))

## 적재 (Ch 6의 `run` 필요) — 가드레일 없이 돌려보고 관계 타입이 얼마나 늘어나는지 비교해 보라

In [ ]:
def load(triples):
    for t in triples:
        run("""
        MERGE (s {name: $subj}) SET s:`%s`
        MERGE (o {name: $obj}) SET o:`%s`
        MERGE (s)-[:`%s`]->(o)
        """ % (t["subj_type"], t["obj_type"], t["rel"]),
        subj=t["subj"], obj=t["obj"])

# load(out["triples"])  # Ch 6 노트북의 연결을 먼저 실행